# Agent de vision — CNN sur silhouettes (confidentialité par conception)

**Objectif du projet : surveiller « sans intrusion visuelle directe ».** Ici le CNN ne voit jamais l'image RGB :
on soustrait l'image de la pièce vide et on ne conserve que la **silhouette** binaire de la personne.

Ce notebook :
1. reproduit **exactement** le découpage par séquence du notebook RGB (comparaison équitable) ;
2. estime le fond de la pièce (médiane d'images de l'**apprentissage seul**) et extrait les silhouettes ;
3. vérifie visuellement les silhouettes (étape de contrôle indispensable) ;
4. entraîne un CNN 128×128 mono-canal, l'évalue au niveau image **et** au niveau séquence ;
5. écrit `vision_cnn_silhouette.keras`, `background.npy` et `silhouette_results.json`.

**Mode d'emploi** : exécuter sur Kaggle (même jeu de données que le notebook RGB) ou en local en définissant la variable
d'environnement `UR_FALL_ROOT`. Copier ensuite les 2 fichiers `.keras`/`.npy` dans le dossier `models/` du projet : l'application
passe automatiquement en mode silhouette.

> Ce notebook n'avait pas été exécuté au moment de la livraison du projet ; ses résultats sont à reporter dans le rapport (§ 4.4).

**Interprétation attendue.** Si le score RGB (1,0) reposait sur le décor ou la séquence, la silhouette — qui supprime couleurs,
textures et arrière-plan — donnera un score plus bas : l'écart mesure alors l'effet de raccourci du modèle RGB.

In [ ]:
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd

ON_KAGGLE = Path("/kaggle").exists()
DATA_ROOT = Path(os.environ.get("UR_FALL_ROOT", "/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset"))
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42            # même graine que le notebook RGB
EPOCHS = 15
BATCH_SIZE = 64
THRESHOLD = 0.5
MIN_RUN = 15         # nb d'images consécutives > seuil pour déclarer une SÉQUENCE « chute » (0,5 s à 30 i/s)
RUN_CV = False       # True : validation croisée groupée par séquence (plus long)

assert DATA_ROOT.exists(), f"Jeu de données introuvable : {DATA_ROOT} (définir UR_FALL_ROOT)"
print("Données :", DATA_ROOT, "| sorties :", OUT.resolve())

## 1. Module d'extraction de silhouettes
La cellule suivante écrit `silhouette.py` — **copie exacte** de `training/silhouette.py`, utilisé aussi par l'application (même représentation à l'entraînement et à l'inférence).

In [ ]:
%%writefile silhouette.py
"""Extraction de silhouettes : respect de la vie privée (« sans intrusion visuelle directe »).

Principe : on dispose d'une image de la pièce *vide* (le « fond »). Pour chaque image
reçue, on soustrait ce fond et on ne conserve que la silhouette binaire de la personne.
L'image RGB n'est ni affichée, ni stockée, ni transmise au modèle.

Ce module est utilisé à l'identique par le notebook d'entraînement
(training/train_cnn_silhouette.ipynb) et par l'application (agents/vision_agent.py),
afin que l'entraînement et l'inférence voient exactement la même représentation.
Dépendances : numpy, Pillow, scipy (déjà installé avec scikit-learn).
"""
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
from PIL import Image
from scipy import ndimage as ndi

WORK_SIZE = (160, 120)   # (largeur, hauteur) de calcul
OUT_SIZE = 128           # côté de la silhouette fournie au CNN
DIFF_THRESHOLD = 30.0    # écart de niveau de gris (0-255) à partir duquel un pixel est « avant-plan »
MIN_AREA_FRAC = 0.004    # une composante < 0,4 % de l'image est du bruit
KEEP_RATIO = 0.25        # on garde les composantes >= 25 % de la plus grande (membres isolés)


def to_gray(image) -> np.ndarray:
    """Image (PIL, ndarray ou chemin) -> niveaux de gris float32 de taille WORK_SIZE."""
    if isinstance(image, (str, Path)):
        with Image.open(image) as handle:
            return to_gray(handle.convert("RGB"))
    if not isinstance(image, Image.Image):
        image = Image.fromarray(np.asarray(image))
    gray = image.convert("L").resize(WORK_SIZE, Image.Resampling.BILINEAR)
    return np.asarray(gray, dtype=np.float32)


def compute_background(grays: Iterable[np.ndarray]) -> np.ndarray:
    """Fond = médiane pixel à pixel d'images de la scène où la personne occupe des positions variées."""
    stack = np.stack(list(grays), axis=0)
    if stack.shape[0] < 3:
        raise ValueError("Au moins 3 images sont nécessaires pour estimer un fond")
    return np.median(stack, axis=0).astype(np.float32)


def background_from_paths(paths: Sequence, n_images: int = 300) -> np.ndarray:
    """Échantillonne n_images chemins répartis uniformément et calcule le fond."""
    paths = list(paths)
    idx = np.unique(np.linspace(0, len(paths) - 1, min(n_images, len(paths))).astype(int))
    return compute_background(to_gray(paths[i]) for i in idx)


def foreground_mask(image, background: np.ndarray, threshold: float = DIFF_THRESHOLD) -> np.ndarray:
    """Masque booléen (hauteur x largeur de WORK_SIZE) de la personne."""
    gray = to_gray(image)
    if gray.shape != background.shape:
        raise ValueError(f"Fond {background.shape} incompatible avec l'image {gray.shape}")
    delta = gray - background
    delta = delta - np.median(delta)               # compense un changement global d'éclairage
    diff = ndi.gaussian_filter(np.abs(delta), 1.0)
    mask = diff > threshold
    mask = ndi.binary_opening(mask, structure=np.ones((3, 3)))
    mask = ndi.binary_closing(mask, structure=np.ones((5, 5)), iterations=2)
    mask = ndi.binary_fill_holes(mask)
    labels, count = ndi.label(mask)
    if count == 0:
        return np.zeros_like(mask, dtype=bool)
    sizes = np.asarray(ndi.sum(mask, labels, index=np.arange(1, count + 1)))
    biggest = sizes.max()
    if biggest < MIN_AREA_FRAC * mask.size:
        return np.zeros_like(mask, dtype=bool)
    keep = [i + 1 for i, s in enumerate(sizes) if s >= KEEP_RATIO * biggest]
    return np.isin(labels, keep)


def to_silhouette(image, background: np.ndarray, threshold: float = DIFF_THRESHOLD) -> np.ndarray:
    """Silhouette uint8 (OUT_SIZE x OUT_SIZE), valeurs 0..255."""
    mask = foreground_mask(image, background, threshold)
    pic = Image.fromarray((mask * 255).astype(np.uint8))
    pic = pic.resize((OUT_SIZE, OUT_SIZE), Image.Resampling.BILINEAR)
    return np.asarray(pic, dtype=np.uint8)


def to_model_input(silhouette: np.ndarray) -> np.ndarray:
    """(OUT_SIZE, OUT_SIZE) uint8 -> (1, OUT_SIZE, OUT_SIZE, 1) float32 valeurs 0..255.

    Le CNN contient sa propre couche Rescaling(1/255), le même tableau est donc
    utilisé à l'entraînement (uint8 converti en float) et à l'inférence.
    """
    return silhouette.astype("float32")[None, ..., None]


def foreground_ratio(silhouette: np.ndarray) -> float:
    """Part de pixels d'avant-plan (outil de diagnostic)."""
    return float((silhouette > 127).mean())

In [ ]:
import silhouette as sil
print('Silhouette :', sil.OUT_SIZE, 'x', sil.OUT_SIZE, '| seuil de différence', sil.DIFF_THRESHOLD)

## 2. Découpage par séquence (identique au notebook RGB)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

png_paths = sorted(DATA_ROOT.rglob("*.png"))
rows = []
for p in png_paths:
    seq_id = p.parent.name.lower()
    if seq_id.startswith("fall"):
        label, label_name = 1, "fall"
    elif seq_id.startswith("adl"):
        label, label_name = 0, "adl"
    else:
        continue
    rows.append({"path": str(p), "seq_id": seq_id, "label": label, "label_name": label_name})
df = pd.DataFrame(rows)
print("Images :", len(df), "| séquences :", df["seq_id"].nunique())

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_val_idx, test_idx = next(gss_test.split(df, df["label"], groups=df["seq_id"]))
train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(gss_val.split(train_val_df, train_val_df["label"], groups=train_val_df["seq_id"]))
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

assert set(train_df.seq_id).isdisjoint(val_df.seq_id) and set(train_df.seq_id).isdisjoint(test_df.seq_id) \
    and set(val_df.seq_id).isdisjoint(test_df.seq_id)
print({n: (len(d), d.seq_id.nunique()) for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]})

# Comparaison avec le découpage du notebook RGB (si le manifeste est disponible à côté du notebook)
manifest = next((m for m in (Path("ur_fall_manifest.csv"), Path("training/ur_fall_manifest.csv"),
                             Path("/kaggle/working/ur_fall_manifest.csv")) if m.exists()), None)
if manifest is not None:
    ref = pd.read_csv(manifest).drop_duplicates("seq_id").set_index("seq_id")["split"]
    mine = {**{s: "train" for s in train_df.seq_id}, **{s: "val" for s in val_df.seq_id}, **{s: "test" for s in test_df.seq_id}}
    same = all(ref.get(s) == v for s, v in mine.items())
    print("Découpage identique au notebook RGB :", same)
    assert same, "Les séquences de test diffèrent de l'expérience RGB : comparaison non équitable"

## 3. Fond de la pièce et silhouettes
Le fond est la médiane de 300 images **de l'apprentissage uniquement**, prises dans des séquences où la personne est à des endroits variés.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **k: x

t0 = time.time()
BACKGROUND = sil.background_from_paths(train_df["path"].tolist(), n_images=300)
print("Fond estimé en %.0f s, forme %s" % (time.time() - t0, BACKGROUND.shape))

def extract(frame_df):
    with ThreadPoolExecutor(max_workers=4) as pool:
        out = list(tqdm(pool.map(lambda p: sil.to_silhouette(p, BACKGROUND), frame_df["path"]), total=len(frame_df)))
    return np.stack(out)[..., None]          # uint8 (N, 128, 128, 1)

t0 = time.time()
X_train, X_val, X_test = extract(train_df), extract(val_df), extract(test_df)
y_train, y_val, y_test = (d["label"].values.astype("float32") for d in (train_df, val_df, test_df))
print("Silhouettes extraites en %.0f s :" % (time.time() - t0), X_train.shape, X_val.shape, X_test.shape)

## 4. Contrôle visuel des silhouettes (à regarder avant d'entraîner)
Les silhouettes doivent ressembler à une personne, pas à du bruit ni à une grande tache. Si la part d'avant-plan est aberrante, ajuster `DIFF_THRESHOLD` dans `silhouette.py` ou estimer un fond par scène.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

ratios = pd.Series([sil.foreground_ratio(s[..., 0]) for s in X_train], index=train_df.index)
by_class = ratios.groupby(train_df["label_name"]).describe()[["mean", "50%", "min", "max"]]
print("Part d'avant-plan par classe (apprentissage) :\n", by_class.round(3).to_string())
med = float(ratios.median())
if not 0.003 < med < 0.35:
    print("ATTENTION : part d'avant-plan médiane %.3f anormale -> silhouettes probablement mauvaises (voir la grille)." % med)
gap = abs(by_class.loc["adl", "mean"] - by_class.loc["fall", "mean"])
print("Écart de part d'avant-plan entre classes : %.3f (un écart énorme signalerait un décor différent entre classes)" % gap)

rng = np.random.default_rng(0)
picks = [rng.choice(np.where(train_df["label"].values == c)[0], 6, replace=False) for c in (0, 1)]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for r, idxs in enumerate(picks):
    for c, i in enumerate(idxs):
        axes[r, c].imshow(X_train[i, ..., 0], cmap="gray"); axes[r, c].axis("off")
        axes[r, c].set_title(f"{train_df.seq_id[i][:6]} #{i}", fontsize=8)
axes[0, 0].set_ylabel("ADL"); axes[1, 0].set_ylabel("chute")
plt.suptitle("Silhouettes (ligne 1 : activités quotidiennes, ligne 2 : séquences de chute)")
plt.tight_layout(); plt.savefig(OUT / "silhouette_samples.png", dpi=120); plt.show()

## 5. Modèle (petit CNN mono-canal 128×128)
Augmentations actives à l'entraînement seulement (miroir horizontal, petite translation). La mise à l'échelle 1/255 est **dans** le modèle : l'application lui fournit directement des valeurs 0-255.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
w = compute_class_weight("balanced", classes=classes, y=y_train.astype(int))
class_weight = {int(c): float(v) for c, v in zip(classes, w)}
print("Poids de classes (0=adl, 1=fall) :", class_weight)

def build_model():
    inputs = layers.Input(shape=(sil.OUT_SIZE, sil.OUT_SIZE, 1))
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomTranslation(0.05, 0.05, fill_mode="constant")(x)
    for filters in (16, 32, 64):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                           tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")])
    return model

tf.keras.utils.set_random_seed(SEED)
model = build_model()
model.summary()

## 6. Entraînement
Arrêt anticipé sur `val_loss` (plus fin que `val_auc`, qui plafonne à 1,0 dès la 2ᵉ époque dans l'expérience RGB), meilleurs poids restaurés.

In [ ]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=4, restore_best_weights=True)]
t0 = time.time()
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE,
                    class_weight=class_weight, callbacks=callbacks, verbose=2)
print("Entraînement : %.0f s, %d époques" % (time.time() - t0, len(history.history["loss"])))

model.save(OUT / "vision_cnn_silhouette.keras")
np.save(OUT / "background.npy", BACKGROUND)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(history.history["loss"], label="apprentissage"); ax[0].plot(history.history["val_loss"], label="validation")
ax[0].set_title("Perte"); ax[0].set_xlabel("époque"); ax[0].legend()
ax[1].plot(history.history["auc"], label="apprentissage"); ax[1].plot(history.history["val_auc"], label="validation")
ax[1].set_title("AUC"); ax[1].set_xlabel("époque"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / "silhouette_training_curves.png", dpi=120); plt.show()

## 7. Évaluation sur les 14 séquences de test
Deux niveaux : **image** (comme le notebook RGB) et **séquence** (une séquence est déclarée « chute » si au moins `MIN_RUN` images consécutives dépassent le seuil).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

probs = model.predict(X_test, batch_size=128, verbose=0).ravel()
pred = (probs > THRESHOLD).astype(int)
yt = y_test.astype(int)
frame = {"accuracy": float(accuracy_score(yt, pred)), "auc": float(roc_auc_score(yt, probs)),
         "confusion_matrix": confusion_matrix(yt, pred).tolist(),
         "report": classification_report(yt, pred, target_names=["adl", "fall"], output_dict=True, zero_division=0)}
print("NIVEAU IMAGE :", {k: round(frame[k], 4) for k in ("accuracy", "auc")})
print(classification_report(yt, pred, target_names=["adl", "fall"], zero_division=0))
print("Matrice de confusion [[VN, FP], [FN, VP]] :", frame["confusion_matrix"])

def longest_run(flags):
    best = cur = 0
    for f in flags:
        cur = cur + 1 if f else 0
        best = max(best, cur)
    return best

eval_df = test_df.assign(prob=probs).sort_values("path")
seq_rows = []
for seq, g in eval_df.groupby("seq_id"):
    run = longest_run((g["prob"] > THRESHOLD).tolist())
    seq_rows.append({"seq_id": seq, "true": g["label_name"].iloc[0], "frames": len(g), "mean_prob": round(float(g["prob"].mean()), 3),
                     "longest_run": run, "predicted": "fall" if run >= MIN_RUN else "adl"})
seq_df = pd.DataFrame(seq_rows)
seq_df["ok"] = seq_df["true"] == seq_df["predicted"]
print("\nNIVEAU SÉQUENCE (14 séquences) :")
print(seq_df.to_string(index=False))
seq_acc = float(seq_df["ok"].mean())
print("Exactitude par séquence : %.1f %%" % (100 * seq_acc))

fig, ax = plt.subplots(figsize=(3.6, 3.2))
cm = np.array(frame["confusion_matrix"]); ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, int(v), ha="center", va="center")
ax.set_xticks([0, 1], ["adl", "fall"]); ax.set_yticks([0, 1], ["adl", "fall"])
ax.set_xlabel("prédit"); ax.set_ylabel("réel"); ax.set_title("Test (images)")
plt.tight_layout(); plt.savefig(OUT / "silhouette_confusion.png", dpi=120); plt.show()

In [ ]:
results = {
    "representation": "silhouette", "input_size": sil.OUT_SIZE, "diff_threshold": sil.DIFF_THRESHOLD,
    "split": {"train_frames": len(train_df), "val_frames": len(val_df), "test_frames": len(test_df),
              "test_sequences": int(test_df.seq_id.nunique()), "same_as_rgb_split": bool(manifest is not None and same)},
    "epochs_run": len(history.history["loss"]), "frame_level": frame,
    "sequence_level": {"min_run": MIN_RUN, "accuracy": seq_acc, "table": seq_df.drop(columns="ok").to_dict(orient="records")},
    "foreground_ratio_median_train": med,
    "reference_rgb_test": {"accuracy": 1.0, "auc": 1.0, "note": "résultat enregistré dans train_cnn.ipynb"},
}
(OUT / "silhouette_results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print("Fichiers à copier dans le dossier models/ du projet :")
for name in ("vision_cnn_silhouette.keras", "background.npy"):
    print("  ", OUT / name)
print("À reporter dans le rapport : ", OUT / "silhouette_results.json")

## 8. (Optionnel) Validation croisée groupée par séquence
Mesure la variabilité due au choix des séquences de test (5 plis, séquences jamais partagées entre apprentissage et test).

In [ ]:
if RUN_CV:
    from sklearn.model_selection import GroupKFold
    X_all = np.concatenate([X_train, X_val]); y_all = np.concatenate([y_train, y_val]).astype(int)
    groups = np.concatenate([train_df["seq_id"].values, val_df["seq_id"].values])
    accs, aucs = [], []
    for k, (tr, te) in enumerate(GroupKFold(n_splits=5).split(X_all, y_all, groups), 1):
        tf.keras.utils.set_random_seed(SEED + k)
        m = build_model()
        m.fit(X_all[tr], y_all[tr], epochs=8, batch_size=BATCH_SIZE, class_weight=class_weight, verbose=0)
        p = m.predict(X_all[te], batch_size=128, verbose=0).ravel()
        accs.append(accuracy_score(y_all[te], p > THRESHOLD)); aucs.append(roc_auc_score(y_all[te], p))
        print(f"pli {k}: accuracy {accs[-1]:.3f}  AUC {aucs[-1]:.3f}")
    print("Moyenne ± écart-type : accuracy %.3f ± %.3f | AUC %.3f ± %.3f" % (np.mean(accs), np.std(accs), np.mean(aucs), np.std(aucs)))
else:
    print("RUN_CV = False : cellule ignorée.")